# ISEScan

In [ ]:
# Concatenar os arquivos FASTA das familias de TEs por hospedeiro ###################################################################################

cd /mnt/dados/victor_baca/outputs/cd-hit # Voltando para o diretório cd-hit
mkdir isescan_cdhit_HOST # Criando a pasta para guardar os arquivos concatenados do ISEScan

cd /mnt/dados/victor_baca/outputs/cd-hit/isescan/Fish
cat *.fasta > Fish.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/isescan/Fish/Fish.fasta /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_HOST

cd /mnt/dados/victor_baca/outputs/cd-hit/isescan/Human
cat *.fasta > Human.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/isescan/Human/Human.fasta /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_HOST

cd /mnt/dados/victor_baca/outputs/cd-hit/isescan/Bovine
cat *.fasta > Bovine.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/isescan/Bovine/Bovine.fasta /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_HOST

# Apos criar os arquivos FASTA por hospedeiro, agora vou criar arquivos concatenados com as combinações possíveis entre os hospedeiros
cd /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_HOST 

cat Fish.fasta Human.fasta Bovine.fasta > all_families_isescan.fasta

cat Fish.fasta Human.fasta > Fish_Human_isescan.fasta

cat Fish.fasta Bovine.fasta > Fish_Bovine_isescan.fasta

cat Human.fasta Bovine.fasta > Human_Bovine_isescan.fasta

In [ ]:
# Indo para a pasta onde estão os arquivos FASTA concatenados das famílias do ISEScan
cd /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_HOST
mkdir cd-hit-est # Criando a pasta para guardar os arquivos do CD-HIT-EST

# Adicionando o CD-HIT ao PATH temporariamente 
export PATH=$PATH::/mnt/dados/victor_baca/tools/cdhit

# cd-hit-est versão do CD-HIT para sequências nucleotídicas
nohup cd-hit-est -i Fish.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Fish_cdhit.fasta &

nohup cd-hit-est -i Human.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Human_cdhit.fasta &

nohup cd-hit-est -i Bovine.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Bovine_cdhit.fasta & 

nohup cd-hit-est -i all_families_isescan.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/all_families_isescan_cdhit_HOST.fasta &

nohup cd-hit-est -i Fish_Human_isescan.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Fish_Human_isescan_cdhit_HOST.fasta &

nohup cd-hit-est -i Fish_Bovine_isescan.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Fish_Bovine_isescan_cdhit_HOST.fasta &

nohup cd-hit-est -i Human_Bovine_isescan.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Human_Bovine_isescan_cdhit_HOST.fasta &

In [ ]:
# Criar o script para contar famílias de Insertion Sequences (IS) em arquivo FASTA
nano contar_familias.py

#####################################################################################################################################################
#!/usr/bin/env python3

import re
from collections import Counter
import sys
import os

def parse_fasta_and_count_families(filename):
    """
    Lê arquivo FASTA e conta as famílias de IS encontradas
    
    Args:
        filename: caminho do arquivo FASTA
    
    Returns:
        Counter object com contagem das famílias
    """
    families = []
    
    try:
        with open(filename, 'r') as f:
            for line in f:
                # Processa apenas linhas de cabeçalho (começam com >)
                if line.startswith('>'):
                    # Procura pelo padrão family=NOME_FAMILIA
                    match = re.search(r'family=([^;]+)', line)
                    if match:
                        family_name = match.group(1)
                        families.append(family_name)
    
    except FileNotFoundError:
        print(f"ERRO: Arquivo '{filename}' não encontrado!")
        sys.exit(1)
    except Exception as e:
        print(f"ERRO ao ler arquivo: {e}")
        sys.exit(1)
    
    return Counter(families)

def main():
    # Verifica se o arquivo foi fornecido como argumento
    if len(sys.argv) != 2:
        print("USO: python3 contar_is.py <arquivo.fasta>")
        print("\nExemplo:")
        print("  python3 contar_is.py Bovine_cdhit.fasta")
        print("  python3 contar_is.py /caminho/completo/para/arquivo.fasta")
        sys.exit(1)
    
    filename = sys.argv[1]
    
    # Verifica se o arquivo existe
    if not os.path.isfile(filename):
        print(f"ERRO: Arquivo '{filename}' não encontrado!")
        print(f"Caminho verificado: {os.path.abspath(filename)}")
        sys.exit(1)
    
    print("="*60)
    print("ANÁLISE DE FAMÍLIAS DE INSERTION SEQUENCES (IS)")
    print("="*60)
    print(f"\nArquivo: {filename}")
    print(f"Caminho completo: {os.path.abspath(filename)}\n")
    
    # Faz a contagem
    family_counts = parse_fasta_and_count_families(filename)
    
    if not family_counts:
        print("Nenhuma família de IS encontrada no arquivo!")
        return
    
    # Mostra resultados
    print(f"Total de sequências IS encontradas: {sum(family_counts.values())}")
    print(f"Número de famílias diferentes: {len(family_counts)}\n")
    
    print("-"*60)
    print(f"{'FAMÍLIA':<30} {'QUANTIDADE':>15}")
    print("-"*60)
    
    # Ordena por quantidade (decrescente) e depois por nome
    for family, count in sorted(family_counts.items(), 
                                key=lambda x: (-x[1], x[0])):
        print(f"{family:<30} {count:>15}")
    
    print("-"*60)
    print(f"{'TOTAL':<30} {sum(family_counts.values()):>15}")
    print("="*60)

if __name__ == "__main__":
    main()
#####################################################################################################################################################

chmod +x contar_familias.py

python3 contar_familias.py Fish_cdhit.fasta 
python3 contar_familias.py Human_cdhit.fasta
python3 contar_familias.py Bovine_cdhit.fasta
python3 contar_familias.py Fish_Human_isescan_cdhit_HOST.fasta
python3 contar_familias.py Fish_Bovine_isescan_cdhit_HOST.fasta
python3 contar_familias.py Human_Bovine_isescan_cdhit_HOST.fasta
python3 contar_familias.py all_families_isescan_cdhit_HOST.fasta

# BLASTn

In [ ]:
# Concatenar os arquivos FASTA dos elementos de TEs por hospedeiro ##################################################################################

cd /mnt/dados/victor_baca/outputs/cd-hit # Voltando para o diretório cd-hit
mkdir blastn_cdhit_HOST # Criando a pasta para guardar os arquivos concatenados do BLASTn

cd /mnt/dados/victor_baca/outputs/cd-hit/blastn/Fish
cat *.fasta > Fish.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/blastn/Fish/Fish.fasta /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST

cd /mnt/dados/victor_baca/outputs/cd-hit/blastn/Human
cat *.fasta > Human.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/blastn/Human/Human.fasta /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST

cd /mnt/dados/victor_baca/outputs/cd-hit/blastn/Bovine
cat *.fasta > Bovine.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/blastn/Bovine/Bovine.fasta /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST

# Apos criar os arquivos FASTA por hospedeiro, agora vou criar arquivos concatenados com as combinações possíveis entre os hospedeiros
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST

cat Fish.fasta Human.fasta Bovine.fasta > all_elements_blastn.fasta

cat Fish.fasta Human.fasta > Fish_Human_blastn.fasta

cat Fish.fasta Bovine.fasta > Fish_Bovine_blastn.fasta

cat Human.fasta Bovine.fasta > Human_Bovine_blastn.fasta

In [ ]:
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST # Diretório onde estão os arquivos FASTA concatenados do BLASTn

# Script  Para filtrar arquivos FASTA por TEs específicos, removendo os overlaps ja analisados
nano filter_fasta_tes.py # Copiar o código abaixo

#####################################################################################################################################################
#!/usr/bin/env python3

import os
import re
from pathlib import Path

# Lista de TEs para filtrar
TES = [
    'ISSag9','ISSag3','ISSag5','ISSag2','IS1381A','ISSag8','IS861','ISSag4','ISStin1','ISSag12','ISSpy1','ISCco2','IS1167','ISSag11','ISSmu1','IS1548','IS1216E','IS1167A','ISSg1','IS1550','ISSsu4','ISSth2','ISSa4','IS256','ISLgar1','IS1252','IS6770','IS1194','IS-LL6','IS981','ISLla2','ISSsu5','ISSdy1','ISSdy2','IS200S','ISPsy14','ISPst2','ISPpu17','ISShes10','ISSth1','ISScr1','ISStso1','ISSsu7','ISSsu6','IS1239','ISSpy2','ISSag10','ISLgar5','ISBun2','TnSsu5','Tn5531','Tn551','Tn1546','Tn6246','Tn7114','Tn2670','TnSs1','Tn6674','Tn4001','Tn5393','Tn501','Tn5041','Tn6214','Tn2'
]

def header_contains_te(header, te_list):
    """
    Verifica se o cabeçalho contém algum dos TEs da lista.
    
    Args:
        header: String do cabeçalho FASTA
        te_list: Lista de nomes de TEs
        
    Returns:
        bool: True se contém algum TE, False caso contrário
    """
    header_upper = header.upper()
    for te in te_list:
        # Busca case-insensitive pelo nome do TE
        if te.upper() in header_upper:
            return True
    return False

def filter_fasta(input_file, output_file, te_list):
    """
    Filtra um arquivo FASTA mantendo apenas sequências com TEs específicos.
    
    Args:
        input_file: Caminho do arquivo FASTA de entrada
        output_file: Caminho do arquivo FASTA de saída
        te_list: Lista de nomes de TEs para filtrar
    """
    kept_sequences = 0
    total_sequences = 0
    
    with open(input_file, 'r') as fin, open(output_file, 'w') as fout:
        current_header = None
        current_sequence = []
        keep_sequence = False
        
        for line in fin:
            line = line.rstrip('\n')
            
            if line.startswith('>'):
                # Processar sequência anterior se existir
                if current_header:
                    total_sequences += 1
                    if keep_sequence:
                        fout.write(current_header + '\n')
                        fout.write('\n'.join(current_sequence) + '\n')
                        kept_sequences += 1
                
                # Nova sequência
                current_header = line
                current_sequence = []
                keep_sequence = header_contains_te(line, te_list)
            else:
                # Linha de sequência
                if line.strip():  # Ignorar linhas vazias
                    current_sequence.append(line)
        
        # Processar última sequência
        if current_header:
            total_sequences += 1
            if keep_sequence:
                fout.write(current_header + '\n')
                fout.write('\n'.join(current_sequence) + '\n')
                kept_sequences += 1
    
    return total_sequences, kept_sequences

def main():
    """Função principal para processar todos os arquivos FASTA."""
    
    # Criar diretório de saída
    output_dir = Path('filtered')
    output_dir.mkdir(exist_ok=True)
    
    # Encontrar todos os arquivos .fasta no diretório atual
    fasta_files = list(Path('.').glob('*.fasta'))
    
    if not fasta_files:
        print("❌ Nenhum arquivo .fasta encontrado no diretório atual")
        return
    
    print(f"📂 Encontrados {len(fasta_files)} arquivos FASTA")
    print(f"🔍 Filtrando por {len(TES)} TEs")
    print("=" * 70)
    
    total_kept = 0
    total_processed = 0
    
    for fasta_file in fasta_files:
        # Definir arquivo de saída
        output_file = output_dir / f"{fasta_file.stem}_filter.fasta"
        
        try:
            # Filtrar arquivo
            total_seqs, kept_seqs = filter_fasta(fasta_file, output_file, TES)
            
            percentage = (kept_seqs / total_seqs * 100) if total_seqs > 0 else 0
            
            print(f"✅ {fasta_file.name}")
            print(f"   Total: {total_seqs} | Mantidas: {kept_seqs} ({percentage:.1f}%)")
            print(f"   Saída: {output_file}")
            print()
            
            total_kept += kept_seqs
            total_processed += total_seqs
            
        except Exception as e:
            print(f"❌ Erro ao processar {fasta_file.name}: {e}")
            print()
    
    print("=" * 70)
    print(f"📊 RESUMO FINAL:")
    print(f"   Sequências processadas: {total_processed}")
    print(f"   Sequências mantidas: {total_kept}")
    if total_processed > 0:
        print(f"   Percentual mantido: {total_kept/total_processed*100:.1f}%")
    print(f"   Arquivos gerados em: {output_dir}/")

if __name__ == "__main__":
    main()
#####################################################################################################################################################

# CTRL + O = Salvar
# CTRL + X = Sair 

chmod +x filter_fasta_tes.py # Tornar o script executável 

python3 filter_fasta_tes.py

In [ ]:
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered # Diretório onde estão os arquivos FASTA filtrados do BLASTn

# Script para separar os elementos de Transposons (Tns) e Sequências de Inserção (ISs) de arquivos FASTA em diretórios separados
nano separate_TEs.py

#####################################################################################################################################################
#!/usr/bin/env python3

import os
import re
from pathlib import Path

def create_output_directories():
    """Cria as subpastas Tns e ISs se não existirem"""
    Path("Tns").mkdir(exist_ok=True)
    Path("ISs").mkdir(exist_ok=True)
    print("✓ Diretórios criados: Tns/ e ISs/")

def identify_element_type(header):
    """
    Identifica o tipo de elemento baseado no header.
    Retorna: 'Tn', 'IS', ou None
    """
    # Remove o '>' inicial se houver
    header = header.lstrip('>')
    
    # Padrões mais abrangentes para capturar todos os casos
    # Procura por Tn seguido de letras/números (ex: TnSs1, Tn551, TnSsu5)
    # Procura por In seguido de underscore ou números (ex: In_Tn, In4)
    tn_pattern = r'\b(Tn[A-Za-z0-9]+|In[_0-9])'
    
    # Procura por IS seguido de letras/números (ex: ISSag5, ISCco2, IS1563)
    is_pattern = r'\bIS[A-Za-z0-9]'
    
    # Verifica primeiro por IS (mais específico)
    if re.search(is_pattern, header, re.IGNORECASE):
        return 'IS'
    # Depois verifica por Tn ou In
    elif re.search(tn_pattern, header, re.IGNORECASE):
        return 'Tn'
    
    return None

def process_fasta_file(input_file):
    """
    Processa um arquivo FASTA e separa em Tns e ISs
    """
    base_name = os.path.splitext(os.path.basename(input_file))[0]
    
    # Arquivos de saída
    tn_output = os.path.join("Tns", f"{base_name}_Tns.fasta")
    is_output = os.path.join("ISs", f"{base_name}_ISs.fasta")
    
    # Contadores
    tn_count = 0
    is_count = 0
    other_count = 0
    other_headers = []  # Lista para armazenar headers não classificados
    
    # Abre os arquivos de saída
    with open(tn_output, 'w') as tn_file, \
         open(is_output, 'w') as is_file, \
         open(input_file, 'r') as infile:
        
        current_header = None
        current_sequence = []
        element_type = None
        
        for line in infile:
            line = line.rstrip('\n')
            
            # Nova sequência
            if line.startswith('>'):
                # Salva a sequência anterior se houver
                if current_header and current_sequence:
                    sequence = '\n'.join(current_sequence)
                    
                    if element_type == 'Tn':
                        tn_file.write(f"{current_header}\n{sequence}\n")
                        tn_count += 1
                    elif element_type == 'IS':
                        is_file.write(f"{current_header}\n{sequence}\n")
                        is_count += 1
                    else:
                        other_count += 1
                        other_headers.append(current_header)
                
                # Processa novo header
                current_header = line
                current_sequence = []
                element_type = identify_element_type(line)
                
            else:
                # Adiciona linha de sequência
                if line.strip():  # Ignora linhas vazias
                    current_sequence.append(line)
        
        # Salva a última sequência
        if current_header and current_sequence:
            sequence = '\n'.join(current_sequence)
            
            if element_type == 'Tn':
                tn_file.write(f"{current_header}\n{sequence}\n")
                tn_count += 1
            elif element_type == 'IS':
                is_file.write(f"{current_header}\n{sequence}\n")
                is_count += 1
            else:
                other_count += 1
                other_headers.append(current_header)
    
    return tn_count, is_count, other_count, other_headers

def main():
    """Função principal"""
    print("="*60)
    print("Separador de Elementos Tns e ISs")
    print("="*60)
    
    # Cria diretórios de saída
    create_output_directories()
    
    # Lista todos os arquivos .fasta no diretório atual
    fasta_files = [f for f in os.listdir('.') if f.endswith('.fasta') or f.endswith('.fa')]
    
    if not fasta_files:
        print("\n⚠ Nenhum arquivo FASTA encontrado no diretório atual!")
        return
    
    print(f"\n✓ Encontrados {len(fasta_files)} arquivo(s) FASTA\n")
    
    # Processa cada arquivo
    total_tn = 0
    total_is = 0
    total_other = 0
    all_other_headers = []  # Lista para todos os headers não classificados
    
    for fasta_file in sorted(fasta_files):
        print(f"Processando: {fasta_file}")
        tn_count, is_count, other_count, other_headers = process_fasta_file(fasta_file)
        
        print(f"  ├─ Transposons (Tns): {tn_count}")
        print(f"  ├─ Seq. Inserção (ISs): {is_count}")
        print(f"  └─ Outros elementos: {other_count}")
        
        # Se houver outros elementos, mostra os headers
        if other_count > 0:
            print(f"\n  ⚠ Headers não classificados em {fasta_file}:")
            for header in other_headers:
                print(f"     {header}")
            all_other_headers.extend([(fasta_file, header) for header in other_headers])
        
        print()
        
        total_tn += tn_count
        total_is += is_count
        total_other += other_count
    
    # Resumo final
    print("="*60)
    print("RESUMO GERAL")
    print("="*60)
    print(f"Total de Transposons (Tns): {total_tn}")
    print(f"Total de Seq. Inserção (ISs): {total_is}")
    print(f"Total de outros elementos: {total_other}")
    
    # Mostra resumo dos elementos não classificados
    if total_other > 0:
        print("\n" + "="*60)
        print("ELEMENTOS NÃO CLASSIFICADOS - RESUMO")
        print("="*60)
        for fasta_file, header in all_other_headers:
            print(f"[{fasta_file}] {header}")
    
    print("\n✓ Processamento concluído!")
    print(f"  ├─ Arquivos Tns salvos em: ./Tns/")
    print(f"  └─ Arquivos ISs salvos em: ./ISs/")

if __name__ == "__main__":
    main()
#####################################################################################################################################################

# CTRL + O = Salvar
# CTRL + X = Sair 

chmod +x separate_TEs.py
python3 separate_TEs.py

In [ ]:
# Indo para a pasta de onde estão os arquivos FASTA dos elementos Transposons (Tns)
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered/Tns
mkdir cd-hit-est # Criando a pasta para guardar os arquivos do CD-HIT-EST

# Adicionando o CD-HIT ao PATH temporariamente 
export PATH=$PATH::/mnt/dados/victor_baca/tools/cdhit

# cd-hit-est versão do CD-HIT para sequências nucleotídicas
nohup cd-hit-est -i Fish_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Fish_cdhit.fasta &

nohup cd-hit-est -i Human_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Human_cdhit.fasta &

nohup cd-hit-est -i Bovine_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Bovine_cdhit.fasta &

nohup cd-hit-est -i Fish_Human_blastn_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Fish_Human_cdhit.fasta &

nohup cd-hit-est -i Fish_Bovine_blastn_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Fish_Bovine_cdhit.fasta &

nohup cd-hit-est -i Human_Bovine_blastn_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Human_Bovine_cdhit.fasta &

nohup cd-hit-est -i all_elements_blastn_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/all_elements_cdhit.fasta &

In [ ]:
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered/Tns # Diretório onde estão os arquivos FASTA dos elementos Tns do BLASTn

# Script para contar elementos de TEs em arquivos FASTA do BLASTn
nano contar_blastn.py

#####################################################################################################################################################
import re
import sys
from collections import Counter

# Verifica se o arquivo foi fornecido como argumento
if len(sys.argv) < 2:
    print("Uso: python contar_tes.py <caminho_do_arquivo.fasta>")
    print("Exemplo: python contar_tes.py /home/user/dados/Human_cdhit.fasta")
    sys.exit(1)

# Obtém o caminho do arquivo do argumento
filename = sys.argv[1]

# Dicionário para contar os TEs
te_counts = Counter()

# Verifica se o arquivo existe e processa
try:
    with open(filename, 'r') as file:
        for line in file:
            # Processa apenas as linhas de cabeçalho (começam com >)
            if line.startswith('>'):
                # Extrai o nome do TE do cabeçalho
                # Formato esperado: >blast_hit_XX ... Name=TE_NAME;...
                match = re.search(r'Name=([^;]+)', line)
                if match:
                    te_name = match.group(1)
                    te_counts[te_name] += 1

except FileNotFoundError:
    print(f"Erro: Arquivo '{filename}' não encontrado!")
    print("Verifique se o caminho está correto.")
    sys.exit(1)
except Exception as e:
    print(f"Erro ao processar o arquivo: {e}")
    sys.exit(1)

# Verifica se algum TE foi encontrado
if not te_counts:
    print("Aviso: Nenhum elemento transponível encontrado no arquivo.")
    print("Verifique se o arquivo está no formato correto.")
    sys.exit(0)

# Exibe os resultados
print("=" * 60)
print("ELEMENTOS TRANSPONÍVEIS ENCONTRADOS")
print("=" * 60)
print(f"\nTotal de TEs únicos: {len(te_counts)}")
print(f"Total de ocorrências: {sum(te_counts.values())}\n")

print("-" * 60)
print(f"{'Nome do TE':<40} {'Quantidade':>10}")
print("-" * 60)

# Ordena por quantidade (decrescente) e depois por nome
for te_name, count in sorted(te_counts.items(), key=lambda x: (-x[1], x[0])):
    print(f"{te_name:<40} {count:>10}")

print("-" * 60)

# Estatísticas adicionais
print("\n" + "=" * 60)
print("ESTATÍSTICAS")
print("=" * 60)
print(f"TE mais frequente: {te_counts.most_common(1)[0][0]} ({te_counts.most_common(1)[0][1]} ocorrências)")
print(f"Média de ocorrências por TE: {sum(te_counts.values()) / len(te_counts):.2f}")
#####################################################################################################################################################

# CTRL + O -> para salvar o arquivo
# CTRL + X -> para sair do editor

chmod +x contar_blastn.py

python contar_blastn.py Fish_cdhit.fasta

python contar_blastn.py Human_cdhit.fasta

python contar_blastn.py Bovine_cdhit.fasta

python contar_blastn.py Fish_Human_cdhit.fasta

python contar_blastn.py Fish_Bovine_cdhit.fasta

python contar_blastn.py Human_Bovine_cdhit.fasta

python contar_blastn.py all_elements_cdhit.fasta

In [ ]:
# Como o elemento ISPa12_ISPpu17 ainda não foi filtrado por ter o ISPpu17 (ISPpu17 um elemento desejado, mais não o ISPa12_ISPpu17), vou criar um novo script para remover/filtrar esse elemento ISPa12_ISPpu17 dos meus arquivos fastas dos elementos ISs

cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered/ISs # Diretório onde estão os arquivos FASTA dos elementos ISs do BLASTn

nano filter_isppa12.py # Copiar o código abaixo

#####################################################################################################################################################
#!/usr/bin/env python3
"""
Script para filtrar sequências FASTA que contêm ISPa12_ISPpu17 no campo Name.
Remove essas sequências de todos os arquivos FASTA no diretório especificado
e salva os resultados em uma nova subpasta.
"""

import os
import re
from pathlib import Path

def parse_fasta(file_path):
    """
    Lê um arquivo FASTA e retorna uma lista de tuplas (header, sequence).
    """
    sequences = []
    current_header = None
    current_seq = []
    
    with open(file_path, 'r') as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                # Salva a sequência anterior se existir
                if current_header is not None:
                    sequences.append((current_header, ''.join(current_seq)))
                # Inicia nova sequência
                current_header = line
                current_seq = []
            else:
                current_seq.append(line)
        
        # Salva a última sequência
        if current_header is not None:
            sequences.append((current_header, ''.join(current_seq)))
    
    return sequences

def has_isppa12_isppu17(header):
    """
    Verifica se o header contém ISPa12_ISPpu17 no campo Name.
    """
    # Procura por Name=...ISPa12_ISPpu17...
    match = re.search(r'Name=([^;]+)', header)
    if match:
        name_value = match.group(1)
        if 'ISPa12_ISPpu17' in name_value:
            return True
    return False

def filter_fasta(input_file, output_file):
    """
    Filtra um arquivo FASTA removendo sequências com ISPa12_ISPpu17 no Name.
    Retorna o número de sequências removidas e mantidas.
    """
    sequences = parse_fasta(input_file)
    
    filtered_sequences = []
    removed_count = 0
    
    for header, seq in sequences:
        if has_isppa12_isppu17(header):
            removed_count += 1
        else:
            filtered_sequences.append((header, seq))
    
    # Escreve o arquivo filtrado
    with open(output_file, 'w') as f:
        for header, seq in filtered_sequences:
            f.write(f"{header}\n{seq}\n")
    
    return removed_count, len(filtered_sequences)

def main():
    # Diretório de entrada
    input_dir = Path('/mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered/ISs')
    
    # Cria o diretório de saída
    output_dir = input_dir / 'filtered_no_ISPa12_ISPpu17'
    output_dir.mkdir(exist_ok=True)
    
    print(f"Diretório de entrada: {input_dir}")
    print(f"Diretório de saída: {output_dir}")
    print("-" * 70)
    
    # Processa todos os arquivos .fasta no diretório
    fasta_files = list(input_dir.glob('*.fasta'))
    
    if not fasta_files:
        print("Nenhum arquivo FASTA encontrado no diretório!")
        return
    
    total_removed = 0
    total_kept = 0
    
    for fasta_file in fasta_files:
        print(f"\nProcessando: {fasta_file.name}")
        
        output_file = output_dir / fasta_file.name
        
        try:
            removed, kept = filter_fasta(fasta_file, output_file)
            total_removed += removed
            total_kept += kept
            
            print(f"  ✓ Sequências removidas: {removed}")
            print(f"  ✓ Sequências mantidas: {kept}")
            print(f"  ✓ Arquivo salvo: {output_file.name}")
            
        except Exception as e:
            print(f"  ✗ Erro ao processar {fasta_file.name}: {e}")
    
    print("\n" + "=" * 70)
    print(f"RESUMO:")
    print(f"  Arquivos processados: {len(fasta_files)}")
    print(f"  Total de sequências removidas: {total_removed}")
    print(f"  Total de sequências mantidas: {total_kept}")
    print(f"  Arquivos salvos em: {output_dir}")
    print("=" * 70)

if __name__ == "__main__":
    main()
#####################################################################################################################################################

# CTRL + O -> para salvar o arquivo
# CTRL + X -> para sair do editor

chmod +x filter_isppa12.py

python3 filter_isppa12.py

In [ ]:
# Indo para a pasta de onde estão os arquivos FASTA dos elementos Sequências de Inserção (ISs)
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered/ISs/filtered_no_ISPa12_ISPpu17
mkdir cd-hit-est # Criando a pasta para guardar os arquivos do CD-HIT-EST

# Adicionando o CD-HIT ao PATH temporariamente 
export PATH=$PATH::/mnt/dados/victor_baca/tools/cdhit

# cd-hit-est versão do CD-HIT para sequências nucleotídicas
nohup cd-hit-est -i Fish_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Fish_cdhit.fasta &

nohup cd-hit-est -i Human_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Human_cdhit.fasta &

nohup cd-hit-est -i Bovine_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Bovine_cdhit.fasta &

nohup cd-hit-est -i Fish_Human_blastn_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Fish_Human_cdhit.fasta &

nohup cd-hit-est -i Fish_Bovine_blastn_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Fish_Bovine_cdhit.fasta &

nohup cd-hit-est -i Human_Bovine_blastn_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Human_Bovine_cdhit.fasta &

nohup cd-hit-est -i all_elements_blastn_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/all_elements_cdhit.fasta &

In [ ]:
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered/ISs/filtered_no_ISPa12_ISPpu17/cd-hit-est # Diretório onde estão os arquivos FASTA dos elementos ISs do BLASTn

# Copiando o script contar_blastn.py para a pasta dos ISs, para poder fazer a contagem dos elementos de ISs
cp /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered/Tns/cd-hit-est/contar_blastn.py /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered/ISs/filtered_no_ISPa12_ISPpu17/cd-hit-est

# Utilizando o script contar_blastn.py para contar os elementos de ISs nos arquivos FASTA do CD-HIT-EST

python contar_blastn.py Fish_cdhit.fasta

python contar_blastn.py Human_cdhit.fasta

python contar_blastn.py Bovine_cdhit.fasta

python contar_blastn.py Fish_Human_cdhit.fasta

python contar_blastn.py Fish_Bovine_cdhit.fasta

python contar_blastn.py Human_Bovine_cdhit.fasta

python contar_blastn.py all_elements_cdhit.fasta
